### Установка зависимостей

In [2]:
# https://microsoft.github.io/graphrag/get_started/
# https://developers.llamaindex.ai/python/framework/getting_started/starter_example_local/
%pip install raglite lightrag-hku[api] scikit-learn ollama tqdm PyMuPDF numpy

  Using cached raglite-1.0.0-py3-none-any.whl.metadata (24 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached ollama-0.6.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached pymupdf-1.26.7-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (3.4 kB)
  Using cached lightrag_hku-1.4.9.10-py3-none-any.whl.metadata (89 kB)
  Using cached duckdb_engine-0.17.0-py3-none-any.whl.metadata (8.4 kB)
  Using cached duckdb-1.4.3-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl.metadata (4.3 kB)
  Using cached fastmcp-2.14.1-py3-none-any.whl.metadata (20 kB)
  Using cached huggingface_hub-1.2.3-py3-none-any.whl.metadata (13 kB)
  Using cached langdetect-1.0.9-py3-none-any.whl
  Using cached litellm-1.80.11-py3-none-any.whl.metadata (29 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux

### Подготовка функций для оценки.
Взято [отсюда](https://www.geeksforgeeks.org/nlp/evaluation-metrics-for-retrieval-augmented-generation-rag-systems/).

In [3]:
import numpy as np

# MRR
def mean_reciprocal_rank(y_true, y_pred):
    reciprocal_ranks = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        rr = 0
        for rank, doc in enumerate(pred_docs, start=1):
            if doc in true_docs:
                rr = 1 / rank
                break
        reciprocal_ranks.append(rr)
    return sum(reciprocal_ranks) / len(reciprocal_ranks)
def ndcg(y_true, y_pred, k=5):
    ndcg_scores = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        pred_docs_k = pred_docs[:k]
        dcg = sum([1 / np.log2(idx + 2) if doc in true_docs else 0 for idx, doc in enumerate(pred_docs_k)])
        ideal_docs_k = true_docs[:k]
        idcg = sum([1 / np.log2(idx + 2) for idx, _ in enumerate(ideal_docs_k)])
        ndcg_scores.append(dcg / idcg if idcg > 0 else 0)
    return np.mean(ndcg_scores)
def recall_precision_at_k(y_true, y_pred, k=5):
    recall_list = []
    precision_list = []
    for true_docs, pred_docs in zip(y_true, y_pred):
        top_k = pred_docs[:k]
        hits = len([doc for doc in top_k if doc in true_docs])
        recall_list.append(hits / len(true_docs) if true_docs else 0)
        precision_list.append(hits / k)
    return np.mean(recall_list), np.mean(precision_list)

# Example usage:
#y_true = [['doc1', 'doc2'], ['doc3']]
#y_pred = [['doc2', 'doc4'], ['doc5']]
#print("nDCG@5:", ndcg(y_true, y_pred))
#will print: nDCG@5: 0.3065735963827292

### Подготовка данных (фильтрация)
Для повышения качества оценки будут использованы данные, экспортированные в MarkDown формат (формулы в TeX-формате) через PaddleOCR (PPv3).
Данные представляют собой статьи по математике из открытого [источника](https://huggingface.co/datasets/PleIAs/Math-PDF/blob/main/math_pdf_tars/openalex_math_pdf_tar_31.tar). Большая часть статей в данном датасете позволяют использование экспорта текста без OCR.
Будет взято меньше 50 статей + 5 статей с релевантной темой для запросов.

Список доп. статей:
* https://math.berkeley.edu/~giventh/papers/qkf.pdf
* https://www.ams.org/journals/jams/2014-27-04/S0894-0347-2014-00797-9/S0894-0347-2014-00797-9.pdf
* https://www.cambridge.org/core/services/aop-cambridge-core/content/view/935C492E469B3B107B20F50DFE0C0F64/S2050509424001476a.pdf/a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf
* https://personal.math.vt.edu/lmihalce/QKlectures(MSJ23).pdf
* и W1605366104.pdf из датасета

In [4]:
DATA_DIR="../../data"
RAW_DATA_DIR=f"{DATA_DIR}/openalex_math_pdf_tar_31"
PROCESSED_DATA_DIR=f"{DATA_DIR}/for_rag_2"
MATH_LIMIT=50

In [45]:
from ollama import Client
qwen3_ollama = Client(
    host='http://localhost:11434'
)

def ollama_req_math(text, prompt_ask="Is this text about math?", model='qwen3:1.7b', text_limit=512):
    output_text = ""
    sys_prompt="You are a helpful assistant"
    prompt = f"{prompt_ask} The text to analyze is below:\n--\n{text[:text_limit]}\n--\nReturn only YES or NO answer without ANY explanation."
    for part in qwen3_ollama.generate(model, prompt=prompt, system=sys_prompt, stream=True):
        if part.thinking:
            continue
            print(part.thinking, end='', flush=True)
        output_text += part.response
    return output_text

In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm
import fitz
from pymupdf import FileDataError
import os
import shutil

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
for file in tqdm(list(Path(RAW_DATA_DIR).glob("*.pdf"))):
    text = ""
    try:
        with fitz.open(file) as doc:
            for page in doc:  # iterate the document pages
                text += page.get_text()
    except FileDataError as e:
        print(f"Error reading file {file.name}: {e}")
        continue
    res = ollama_req_math(text=text)
    #print(f"File: {file.name}, Result: {res}")
    if res == "YES":
        # just copy file
        print(f"Copying file: {file.name}")
        shutil.copy(file, os.path.join(PROCESSED_DATA_DIR, file.name))
        if len(math_files) >= MATH_LIMIT:
            print(f"Reached math file limit of {MATH_LIMIT}. Stopping.")
            break
    elif res == "NO":
        print(f"Skipping file: {file.name}")
    else:
        print(f"Unexpected result for file {file.name}: {res}")

    

  0%|          | 0/5000 [00:00<?, ?it/s]

Skipping file: W4295565825.pdf
Copying file: W2957174555_1.pdf
Copying file: W3167731107_2.pdf
Copying file: W2796609034.pdf
Skipping file: W4287119660_1.pdf
Copying file: W4393200428.pdf
Skipping file: W2602179841_3.pdf
Copying file: W4313001346.pdf
Skipping file: W2512409555_1.pdf
Skipping file: W4312320968.pdf
Skipping file: W4386767063.pdf
Copying file: W4377086487_5.pdf
MuPDF error: library error: FT_New_Memory_Face(UBPIRB+CMMI9): broken table

Copying file: W2465613768.pdf
Copying file: W4289128375.pdf
Copying file: W4226456969_2.pdf
Skipping file: W818768447.pdf
Copying file: W4317037187_2.pdf
Copying file: W4396577029.pdf
Copying file: W4283689522.pdf
Skipping file: W3188079605_3.pdf
Copying file: W2164376650_2.pdf
Copying file: W4246716229.pdf
Skipping file: W2791154508.pdf
Error reading file W4394623322.pdf: Failed to open file '../../data/openalex_math_pdf_tar_31/W4394623322.pdf'.
Copying file: W4300932590_1.pdf
Copying file: W2551158135.pdf
Skipping file: W2802454767_1.pdf


In [ ]:
# Now second analyze with more params
for file in tqdm(list(Path(PROCESSED_DATA_DIR).glob("*.pdf"))):
    text = ""
    try:
        with fitz.open(file) as doc:
            for page in doc:  # iterate the document pages
                text += page.get_text()
    except FileDataError as e:
        print(f"Error reading file {file.name}: {e}")
        continue
    res = ollama_req_math(text=text, prompt_ask="Is this text about math and contains formulas?", model='qwen3:8b', text_limit=2048)
    #print(f"File: {file.name}, Result: {res}")
    if res == "YES":
        print(f"Leaving file: {file.name}")
    elif res == "NO":
        print(f"Removing file: {file.name}")
        os.remove(file)
    else:
        print(f"Unexpected result for file {file.name}: {res}")

  0%|          | 0/50 [00:00<?, ?it/s]

Leaving file: W2957174555_1.pdf
Removing file: W3167731107_2.pdf
Leaving file: W2796609034.pdf
Removing file: W4393200428.pdf
Leaving file: W4313001346.pdf
Leaving file: W4377086487_5.pdf
MuPDF error: library error: FT_New_Memory_Face(UBPIRB+CMMI9): broken table

Leaving file: W2465613768.pdf
Leaving file: W4289128375.pdf
Leaving file: W4226456969_2.pdf
Leaving file: W4317037187_2.pdf
Removing file: W4396577029.pdf
Leaving file: W4283689522.pdf
Removing file: W2164376650_2.pdf
Removing file: W4246716229.pdf
Removing file: W4300932590_1.pdf
Removing file: W2551158135.pdf
Removing file: W2223722977_3.pdf
Leaving file: W4206656803.pdf
Removing file: W3157468823.pdf
Leaving file: W4324126587_2.pdf
Removing file: W4297662594.pdf
Leaving file: W1603196302_1.pdf
Leaving file: W2158760784.pdf
Removing file: W2508541584_1.pdf
Leaving file: W2519367019.pdf
Leaving file: W2108242840.pdf
Leaving file: W2999061577_2.pdf
Leaving file: W25036734.pdf
Leaving file: W4285891493.pdf
Removing file: W30190

In [ ]:
files_to_add = [
    "QKlectures(MSJ23).pdf",
    "qkf.pdf",
    "S0894-0347-2014-00797-9.pdf",
    "a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf"
]

OLD_DATA_DIR=f"{DATA_DIR}/for_rag"

for file_name in files_to_add:
    src_path = os.path.join(OLD_DATA_DIR, file_name)
    dst_path = os.path.join(PROCESSED_DATA_DIR, file_name)
    if os.path.exists(src_path):
        print(f"Adding file: {file_name}")
        shutil.copy(src_path, dst_path)
    else:
        raise FileNotFoundError(f"File to add not found: {file_name}")

Adding file: QKlectures(MSJ23).pdf
Adding file: qkf.pdf
Adding file: S0894-0347-2014-00797-9.pdf
Adding file: a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf


In [7]:
from pathlib import Path

math_files = [ file.name for file in list(Path(PROCESSED_DATA_DIR).glob("*.pdf"))]
print(f"Total math-related files prepared for RAG: {len(math_files)}")
for file in math_files:
    print(f"* {file}")

Total math-related files prepared for RAG: 35
* W2957174555_1.pdf
* W2796609034.pdf
* W4313001346.pdf
* W4377086487_5.pdf
* W2465613768.pdf
* W4289128375.pdf
* W4226456969_2.pdf
* W4317037187_2.pdf
* W4283689522.pdf
* W4206656803.pdf
* W4324126587_2.pdf
* W1603196302_1.pdf
* W2158760784.pdf
* W2519367019.pdf
* W2108242840.pdf
* W2999061577_2.pdf
* W25036734.pdf
* W4285891493.pdf
* W2945171940.pdf
* W2171382235.pdf
* W3118338062.pdf
* W4319655444_7.pdf
* W3211598426.pdf
* W3152618620_1.pdf
* W2963449700_2.pdf
* W4394719147.pdf
* W4310022274_2.pdf
* W4376956170.pdf
* W4386002119.pdf
* W2972953549_2.pdf
* W3128183679.pdf
* S0894-0347-2014-00797-9.pdf
* QKlectures(MSJ23).pdf
* a-presentation-of-the-torus-equivariant-quantum-k-theory-ring-of-flag-manifolds-of-type-a-part-ii-quantum-double-grothendieck-polynomials.pdf
* qkf.pdf
